# GLProtein + Global Structure Triplet Loss

This notebook sets up the environment, unzips the uploaded repo zip, installs dependencies and provides commands to test GLProtein with global structure triplet loss.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Mount Google Drive
from google.colab import drive, files
import os

drive.mount('/content/drive')


# Required files in your drive, under folder GLProtein_TMVec:
# (1) Repo zip: GLProtein_TMVec.zip
# (2) SwissProt sequences: swissprot_seq.fasta (for triplet generation test)
# (3) TM-Vec embeddings of SwissProt: swiss_large.npy (for triplet generation test)
# (4) Triplet TSV: tmvec_triplets_small.tsv (for pre-training test)
REPO_ZIP_PATH     = "/content/drive/MyDrive/GLProtein_TMVec/GLProtein_TMVec.zip"
TMVEC_FASTA_PATH    = "/content/drive/MyDrive/GLProtein_TMVec/swissprot_seq.fasta"
TMVEC_NPY_PATH    = "/content/drive/MyDrive/GLProtein_TMVec/swiss_large.npy"
TMVEC_TSV_PATH    = "/content/drive/MyDrive/GLProtein_TMVec/tmvec_triplets_small.tsv"

# Locations for assets inside the repo after unzip
PRETRAIN_DATA_DIR = "data/pretrain_data"

def _exists(p: str) -> bool:
    return p and os.path.exists(p)

print("REPO_ZIP_PATH exists:", _exists(REPO_ZIP_PATH), REPO_ZIP_PATH)
print("TMVEC_FASTA_PATH exists:", _exists(TMVEC_FASTA_PATH), TMVEC_FASTA_PATH)
print("TMVEC_NPY_PATH exists:", _exists(TMVEC_NPY_PATH), TMVEC_NPY_PATH)
print("TMVEC_TSV_PATH exists:", _exists(TMVEC_TSV_PATH), TMVEC_TSV_PATH)

if not _exists(REPO_ZIP_PATH):
    print("Please upload the repo zip")
    uploaded = files.upload()
    print("Uploaded:", list(uploaded.keys()))
    for k in uploaded.keys():
        if k.endswith(".zip"):
            REPO_ZIP_PATH = os.path.abspath(k)
            break
    print("Current repo zip path:", REPO_ZIP_PATH)

In [ ]:
import os, zipfile, glob, shutil

assert 'REPO_ZIP_PATH' in globals(), "Repo zip path is undefined"

local_zip = "/content/repo.zip"
if os.path.abspath(REPO_ZIP_PATH) != local_zip:
    shutil.copy(REPO_ZIP_PATH, local_zip)
else:
    local_zip = REPO_ZIP_PATH

if os.path.exists("repo"):
    shutil.rmtree("repo")
os.makedirs("repo", exist_ok=True)

with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall("repo")

root_candidates = []
for root, dirs, files_ in os.walk("repo"):
    if "run_pretrain_refactor.py" in files_:
        root_candidates.append(root)
assert len(root_candidates) > 0, "No run_pretrain_refactor.py after unzip"

repo_root = root_candidates[0]
print("Repo root:", repo_root)
os.chdir(repo_root)
print("Working directory:", os.getcwd())

In [ ]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install -r requirements.txt

In [ ]:
import os, shutil, gzip

assert 'PRETRAIN_DATA_DIR' in globals(), "Data directory not found"

os.makedirs(PRETRAIN_DATA_DIR, exist_ok=True)

def copy_file(src_path: str, dst_path: str):
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    shutil.copy(src_path, dst_path)

def copy_or_decompress_gz(src_path: str, dst_dir: str):
    """Copy a file into dst_dir. If it's a .gz, decompress into dst_dir."""
    if not src_path or not os.path.exists(src_path):
        return None

    lower = src_path.lower()
    os.makedirs(dst_dir, exist_ok=True)

    if lower.endswith(".gz") and not (lower.endswith(".tar.gz") or lower.endswith(".tgz")):
        out_path = os.path.join(dst_dir, os.path.basename(src_path[:-3]))
        with gzip.open(src_path, "rb") as fin, open(out_path, "wb") as fout:
            shutil.copyfileobj(fin, fout)
        return out_path

    out_path = os.path.join(dst_dir, os.path.basename(src_path))
    shutil.copy(src_path, out_path)
    return out_path

fasta_dst, npy_dst, tsv_dst = None, None, None
if TMVEC_FASTA_PATH and os.path.exists(TMVEC_FASTA_PATH):
    fasta_dst = copy_or_decompress_gz(TMVEC_FASTA_PATH, PRETRAIN_DATA_DIR)
if TMVEC_NPY_PATH and os.path.exists(TMVEC_NPY_PATH):
    npy_dst = copy_or_decompress_gz(TMVEC_NPY_PATH, PRETRAIN_DATA_DIR)
if TMVEC_TSV_PATH and os.path.exists(TMVEC_TSV_PATH):
    tsv_dst = copy_or_decompress_gz(TMVEC_TSV_PATH, PRETRAIN_DATA_DIR)

print("data/pretrain_data:", os.listdir(PRETRAIN_DATA_DIR))

print("Destinations:")
print(".fasta:", fasta_dst)
print(".npy:", npy_dst)
print(".tsv:", tsv_dst)

if fasta_dst is None:
    print("FASTA not copied. Triplet generation will fail.")
if npy_dst is None:
    print("NPY not copied. Triplet generation will fail.")
if tsv_dst is None:
    print("TSV not copied. Run triplet generation before pre-training.")

## (Optional) Triplet Generation Test

If TSV is missing, run this to generate `tmvec_triplets_small.tsv`.

In [ ]:
!python generate_tmvec_pairs_tsv.py \
  --swiss_fasta data/pretrain_data/swissprot_seq.fasta \
  --swiss_tmvec_emb_npy data/pretrain_data/swiss_large.npy \
  --out_tsv data/pretrain_data/tmvec_triplets_small.tsv \
  --out_metadata_json data/pretrain_data/tmvec_triplets_small.metadata.json \
  --top_k_pos 5 \
  --triplets_per_anchor 1 \
  --positive_search_k 64 \
  --negative_tmscore_max 0.2 \
  --negative_pick_strategy hardest \
  --use_faiss \
  --seed 2021 \
  --max_proteins 15000 \
  --log_every_anchors 1000

## Pre-training


In [ ]:
!python run_pretrain_refactor.py \
  --output_dir outputs/glprotein_triplet_small \
  --pretrain_data_dir data/pretrain_data \
  --model_protein_seq_data True \
  --use_tmvec_loss True \
  --tmvec_triplets_tsv tmvec_triplets_small.tsv \
  --weight_decay 0.01 \
  --lr_scheduler_type linear \
  --lm_learning_rate 1e-5 \
  --lm_warmup_ratio 0.167 \
  --fp16 \
  --dataloader_pin_memory \
  --seed 2021 \
  --per_device_train_batch_size 2 \
  --logging_steps 1 \
  --save_steps 10 \
  --max_steps 30 \
  --gradient_accumulation_steps 5 \
  --gradient_checkpointing True \
  --triplet_microbatch_size 1 \
  --max_tokens_per_batch 4096 \
  --max_protein_seq_length 1024

## Full Triplet Generation & Pre-Training

See README for data preparation and arguments. L4 GPU is required.